In [1]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

# ---------------------------------------------------------------------------
# Package imports
# ---------------------------------------------------------------------------
from blackhole.constants import G, M_sun, c
from blackhole.disk_physics import (
    R_func,
    X_func,
    kinematic_viscosity,
)
from blackhole.evolution import (
    add_mass,
    calculate_timestep,
    disk_evap,
    evolve_surface_density,
)
from blackhole.opacity import kappa_bath
from blackhole.solvers import (
    solve_scale_height,
    solve_temperature,
)
from blackhole.viscosity import alpha_visc

# ---------------------------------------------------------------------------
# Black Hole evaporation simulation parameters
# No irradiation, evaporation ENABLED
# ---------------------------------------------------------------------------
alpha_cold = 0.04
alpha_hot = 0.2

M_star = 9 * M_sun  # 9 solar mass black hole (g)

# Disk parameters
R_1 = 5e8    # Inner radius of the disk (cm)
R_K = 2.2e11 # Radius where initial mass is added (cm)
R_N = 4.2e11 # Outer radius of the disk (cm)
M_dot = 1e17 # Mass transfer rate (g/s)

min_Sigma = 1e-5

X_1 = X_func(R_1)
X_K = X_func(R_K)
X_N = X_func(R_N)

# Simulation parameters
N = 100
N_n = 3

# Define the radial grid
X = np.linspace(X_1, X_N, N)
dX = float(np.diff(X)[0])
X = np.linspace(X_1, X_N + N_n * dX, N + N_n)
r = R_func(X)
dr = np.diff(r)
dr = np.insert(dr, 0, 0)
dX = float(np.diff(X)[0])
X_N = X_N + N_n * dX
N = N + N_n

# Initial conditions
Sigma = np.full(N, min_Sigma)
T_c = np.full(N, 1e3)
H = np.full(N, 1e7)
alpha = np.full(N, alpha_cold)

nu = kinematic_viscosity(H, r, alpha, M_star)

j_val = 0
dMj = 0.0
dMj1 = 0.0

# ---------------------------------------------------------------------------
# Timestep strategy
# ---------------------------------------------------------------------------
# Crank-Nicolson (theta=0.5) is unconditionally stable for diffusion, so
# we use a 100x CFL multiplier with a floor of 2000 s.  An adaptive
# thermal-timescale constraint (see simulation loop) automatically reduces
# dt during outbursts when the heating front must be resolved, preventing
# the alpha(T) transition from causing numerical runaway.
dt = calculate_timestep(X, nu, dX) * 100
if dt < 2000:
    dt = 2000

# Eddington luminosity for L_ratio scaling
L_edd = 1.4e38 * (M_star / M_sun)

# Tidal torque parameters
tidal_params = {"cw": 0.2, "a_1": 2.2e12, "n_1": 5, "trunc_frac": 9.1 / 10}


def find_inner_index(Sigma_arr):
    """Find index of first cell with significant surface density."""
    candidates = np.where(Sigma_arr > min_Sigma * 21)[0]
    if len(candidates) == 0:
        candidates = np.where(Sigma_arr > min_Sigma)[0]
    return int(candidates[0]) if len(candidates) > 0 else 0


def plot_s(Sigma_arr, r_arr):
    """Quick plot of a quantity vs radius."""
    plt.figure(figsize=(10, 6))
    plt.plot(np.asarray(r_arr), np.asarray(Sigma_arr))
    plt.show()

In [2]:
timesteps = 100001
output_times = set(np.linspace(1, 100000, 20000, dtype='int64'))

In [3]:
# ---------------------------------------------------------------------------
# Initialize fresh disk
# ---------------------------------------------------------------------------
totalt = 0.0

Sigma_history = [Sigma.copy()]
Temp_history = [T_c.copy()]
H_history = [H.copy()]
alpha_history = [alpha.copy()]
t_history = [0.0]
Sigma_transfer_history = [float(Sigma[1])]

np.savetxt("../data/Sigma_history_bath_array_ev.csv", Sigma_history, delimiter=",")
np.savetxt("../data/Temp_history_bath_array_ev.csv", Temp_history, delimiter=",")
np.savetxt("../data/H_history_bath_array_ev.csv", H_history, delimiter=",")
np.savetxt("../data/t_history_bath_array_ev.csv", t_history, delimiter=",")
np.savetxt("../data/alpha_history_bath_array_ev.csv", alpha_history, delimiter=",")
np.savetxt("../data/Sigma_transfer_history_bath_array_ev.csv", Sigma_transfer_history, delimiter=",")

print(f"Fresh disk initialized: N={N}, Sigma=[{Sigma.min():.2e}, {Sigma.max():.2e}], T_c=[{T_c.min():.0f}, {T_c.max():.0f}] K")
print(f"Initial dt = {dt:.2f} s")
print("Saved to ../data/*_history_bath_array_ev.csv")

Fresh disk initialized: N=103, Sigma=[1.00e-05, 1.00e-05], T_c=[1000, 1000] K
Initial dt = 161511.71 s
Saved to ../data/*_history_bath_array_ev.csv


In [4]:
%%time

#########################################
############### Simulation ##############
#########################################

DATA_PREFIX = "../data/"
DATA_SUFFIX = "_history_bath_array_ev.csv"

trunc_rad = int(N * (9.1 / 10))

for i in range(1):

    Sigma_history = pd.read_csv(DATA_PREFIX + "Sigma" + DATA_SUFFIX, header=None).to_numpy().tolist()
    Temp_history = pd.read_csv(DATA_PREFIX + "Temp" + DATA_SUFFIX, header=None).to_numpy().tolist()
    H_history = pd.read_csv(DATA_PREFIX + "H" + DATA_SUFFIX, header=None).to_numpy().tolist()
    alpha_history = pd.read_csv(DATA_PREFIX + "alpha" + DATA_SUFFIX, header=None).to_numpy().tolist()
    t_history = pd.read_csv(DATA_PREFIX + "t" + DATA_SUFFIX, header=None).to_numpy().flatten().tolist()
    Sigma_transfer_history = pd.read_csv(
        DATA_PREFIX + "Sigma_transfer" + DATA_SUFFIX, header=None
    ).to_numpy().flatten().tolist()

    totalt = t_history[-1]
    Sigma = np.array(Sigma_history[-1])
    T_c = np.array(Temp_history[-1])
    H = np.array(H_history[-1])
    nu = kinematic_viscosity(H, r, alpha, M_star)
    # Crank-Nicolson (theta=0.5) is unconditionally stable for diffusion.
    # Use 100x CFL with floor 2000 s.  The thermal-timescale constraint
    # (below) automatically reduces dt during outbursts.
    dt = calculate_timestep(X, nu, dX) * 100
    if dt < 2000:
        dt = 2000

    for timestep in range(timesteps):

        # Stage 1: Add mass at the outer radius
        result = add_mass(Sigma, M_dot, dt, X, N, X_K, X_N, dX, min_Sigma)
        Sigma = result.Sigma
        j_val = result.j_val
        dMj = result.dMj
        dMj1 = result.dMj1

        # Compute luminosity-scaled evaporation (Liu et al. 2002):
        # L_ratio = L_actual / L_edd, capped at 1.0
        M_dot_inner = 6.0 * np.pi * Sigma[1] * nu[1]
        L_actual = 0.1 * M_dot_inner * c**2
        L_ratio = min(L_actual / L_edd, 1.0) if L_edd > 0 else 0.0

        def evap_func(r_arr, _lr=L_ratio):
            return disk_evap(r_arr, M_star, L_ratio=_lr)

        # Stage 2: Evolve surface density (diffusion + tidal torques + evaporation)
        # Use Crank-Nicolson (theta=0.5) for unconditional stability
        tp = tidal_params if j_val >= trunc_rad else None
        Sigma = evolve_surface_density(
            Sigma, dt, nu, X, dX, N, min_Sigma,
            tidal_params=tp, evap_func=evap_func,
            theta=0.5,
        )

        # Stage 3: Update disk temperature (no irradiation)
        T_c = solve_temperature(H, Sigma, r, T_c, alpha, M_star,
                                opacity_func=kappa_bath)

        # Scale height
        H = solve_scale_height(H, Sigma, r, T_c, M_star)

        # Alpha viscosity (standard temperature-dependent, no irradiation)
        alpha = alpha_visc(T_c, alpha_cold=alpha_cold, alpha_hot=alpha_hot)

        # Update viscosity
        nu = kinematic_viscosity(H, r, alpha, M_star)

        # -----------------------------------------------------------------
        # Adaptive timestep with thermal-timescale constraint
        # -----------------------------------------------------------------
        # Base: 100x CFL with floor 2000 s (covers quiescence efficiently).
        # During outbursts, the alpha(T) transition must be resolved over
        # multiple steps.  We limit dt to 20x the thermal timescale
        # (t_th = 1 / (alpha * Omega_K)) at the hottest cell.
        # In quiescence (all cells cold), this is inactive.
        dt = calculate_timestep(X, nu, dX) * 100
        if dt < 2000:
            dt = 2000

        alpha_max_val = float(np.max(alpha[1:-1]))
        if alpha_max_val > 2.0 * alpha_cold:
            i_hot = int(np.argmax(alpha[1:-1])) + 1
            Omega_K = np.sqrt(G * M_star / r[i_hot]**3)
            t_thermal = 1.0 / (alpha_max_val * Omega_K)
            dt = min(dt, max(20.0 * t_thermal, 2000.0))

        totalt += dt

        # Store state at specific timesteps
        if timestep in output_times:
            Inner_index = find_inner_index(Sigma)
            Sigma_transfer_history.append(float(Sigma[Inner_index]))
            Sigma_history.append(Sigma.copy())
            Temp_history.append(T_c.copy())
            H_history.append(H.copy())
            alpha_history.append(alpha.copy())
            t_history.append(totalt)

    np.savetxt(DATA_PREFIX + "Sigma" + DATA_SUFFIX, Sigma_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "Temp" + DATA_SUFFIX, Temp_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "H" + DATA_SUFFIX, H_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "t" + DATA_SUFFIX, t_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "alpha" + DATA_SUFFIX, alpha_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "Sigma_transfer" + DATA_SUFFIX, Sigma_transfer_history, delimiter=",")

print(f"Simulation complete: {len(t_history)} snapshots, total time = {totalt:.2e} s")

Simulation complete: 20001 snapshots, total time = 8.94e+08 s
CPU times: total: 37.2 s
Wall time: 38 s
